<a href="https://colab.research.google.com/github/miasmita/DM_ACT_2.12_DP/blob/main/IEI097_Lab2_12_MLP_MLflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Actividad 2.12 — Laboratorio: Implementación de Redes Neuronales (MLP + MLflow)

**Nombre:** Daniel Pellegrini
**Número de lista (Semilla / SEED):** 42
**Fecha:** 9 de noviembre de 2026

In [7]:
# Instalar librerías requeridas
!pip install -q mlflow pyngrok kagglehub

import pandas as pd
import numpy as np
import random
import os
import kagglehub
import matplotlib.pyplot as plt

# Definir la semilla (reemplaza 42 por tu número de lista)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import sklearn
import mlflow
print(f"Pandas: {pd.__version__}, Scikit-learn: {sklearn.__version__}, MLflow: {mlflow.__version__}")

Pandas: 2.2.3, Scikit-learn: 1.6.1, MLflow: 3.18.0


In [8]:
from sklearn.model_selection import train_test_split

# Descargar dataset
path = kagglehub.dataset_download("uciml/breast-cancer-wisconsin-data")
csv_path = os.path.join(path, "data.csv") # Asegúrate de que este sea el nombre correcto del CSV
df = pd.read_csv(csv_path)

# Limpieza y codificación
df.drop(columns=['id', 'Unnamed: 32'], inplace=True, errors='ignore')
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

print("Balance de clases:")
print(df['diagnosis'].value_counts())

# Evidencia de escala
variables = ['radius_mean', 'area_mean', 'smoothness_mean']
for var in variables:
    print(f"{var} - Mínimo: {df[var].min()}, Máximo: {df[var].max()}")

# Partición de datos (se mantiene fija en todo el laboratorio)
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

Using Colab cache for faster access to the 'breast-cancer-wisconsin-data' dataset.
Balance de clases:
diagnosis
0    357
1    212
Name: count, dtype: int64
radius_mean - Mínimo: 6.981, Máximo: 28.11
area_mean - Mínimo: 143.5, Máximo: 2501.0
smoothness_mean - Mínimo: 0.05263, Máximo: 0.1634


### Observaciones de los datos

**Balance de clases:**
* (Anota aquí el resultado numérico, por ejemplo: Benigno: 357, Maligno: 212)

**Evidencia de escala:**
Al comparar el mínimo y máximo, se observa que variables como `area_mean` alcanzan valores en los cientos o miles, mientras que `smoothness_mean` maneja rangos decimales muy pequeños. El rango de magnitud de `area_mean` es miles de veces mayor que el de otras variables más pequeñas, lo que evidencia la necesidad de estandarizar.

In [9]:
import os, subprocess, time, requests
from google.colab import userdata
from pyngrok import ngrok

# 1) Levantar el servidor MLflow en segundo plano
env = dict(os.environ,
           MLFLOW_SERVER_ALLOWED_HOSTS="*",
           MLFLOW_SERVER_CORS_ALLOWED_ORIGINS="*")

subprocess.Popen(["mlflow", "server",
                  "--backend-store-uri", "sqlite:///mlflow.db",
                  "--host", "127.0.0.1", "--port", "5000"], env=env)

# 2) Esperar a que responda
for _ in range(60):
    try:
        if requests.get("http://127.0.0.1:5000/health", timeout=1).ok:
            break
    except (requests.ConnectionError, requests.Timeout): # <-- Se agregó requests.Timeout
        pass
    time.sleep(1)

# 3) Túnel público con ngrok
ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))
ngrok.kill() # Cierra túneles anteriores por seguridad
tunel = ngrok.connect(5000)
print("URL de MLflow UI:", tunel.public_url)

# Conectar el notebook a MLflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("IEI097-L2.12-Pellegrini")

URL de MLflow UI: https://catty-grafting-entrap.ngrok-free.dev


2026/11/09 21:32:42 INFO mlflow.tracking.fluent: Experiment with name 'IEI097-L2.12-Pellegrini' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1794259963280, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1794259963280, lifecycle_stage='active', name='IEI097-L2.12-Pellegrini', tags={}, trace_location=None, workspace='default'>

### Hipótesis para Runs A (sin escalar) vs Run B (escalado)

**Hipótesis:** El escalado de las variables (Run B) mejorará drásticamente los resultados y reducirá el número de iteraciones necesarias para converger.
**Justificación:** Las neuronas del MLP calculan sumas ponderadas. Si ingresamos variables con magnitudes enormes (como el área del tumor) junto con variables minúsculas, los gradientes se desestabilizan y la función de activación se satura. Estandarizar (media 0, varianza 1) equilibrará el impacto de todas las características.

In [10]:
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, f1_score, ConfusionMatrixDisplay

def correr_experimento(nombre, escalado, capas, activacion, random_state_mod, tag_fase):
    with mlflow.start_run(run_name=nombre):

        if escalado:
            clf = Pipeline([
                ('scaler', StandardScaler()),
                ('mlp', MLPClassifier(hidden_layer_sizes=capas, activation=activacion, random_state=random_state_mod, max_iter=500))
            ])
        else:
            clf = Pipeline([
                ('mlp', MLPClassifier(hidden_layer_sizes=capas, activation=activacion, random_state=random_state_mod, max_iter=500))
            ])

        clf.fit(X_train, y_train)
        mlp_model = clf.named_steps['mlp']

        y_pred = clf.predict(X_test)

        # MLflow Params & Metrics
        mlflow.log_params({"escalado": escalado, "arquitectura": str(capas), "activacion": activacion, "random_state": random_state_mod, "semilla_particion": SEED})
        mlflow.log_metrics({
            "acc_train": accuracy_score(y_train, clf.predict(X_train)),
            "acc_test": accuracy_score(y_test, y_pred),
            "recall_maligno": recall_score(y_test, y_pred),
            "f1_test": f1_score(y_test, y_pred),
            "n_iter": mlp_model.n_iter_
        })

        for step, loss in enumerate(mlp_model.loss_curve_):
            mlflow.log_metric("train_loss", loss, step=step)

        mlflow.set_tag("fase", tag_fase)

        # Artefacto Matriz de Confusión
        fig, ax = plt.subplots()
        ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, display_labels=['Benigno', 'Maligno'])
        plt.title(f"{nombre}")
        fig.savefig(f"cm_{nombre}.png")
        mlflow.log_artifact(f"cm_{nombre}.png")
        plt.close(fig)

# Ejecutar los 4 runs solicitados
correr_experimento("Run_A", False, (30,), 'relu', SEED, "experimentos")
correr_experimento("Run_B", True,  (30,), 'relu', SEED, "experimentos")
correr_experimento("Run_C", True,  (64, 32), 'relu', SEED, "experimentos")
correr_experimento("Run_D", True,  (64, 32), 'tanh', SEED, "experimentos")

🏃 View run Run_A at: http://127.0.0.1:5000/#/experiments/1/runs/30cae09c4bab49cc909348f657bf9988
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run Run_B at: http://127.0.0.1:5000/#/experiments/1/runs/eea276c2f1864afba3ce11744db99283
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run Run_C at: http://127.0.0.1:5000/#/experiments/1/runs/8ba0d6b77a094850ab37d4981be2c971
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run Run_D at: http://127.0.0.1:5000/#/experiments/1/runs/f83fed57aa594034bd23a20787886624
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


### Interpretación de resultados (Ejecuciones A, B, C y D)

Al observar la tabla de MLflow, la hipótesis se confirma: el Run B (escalado) supera al Run A (sin escalar) en métricas de clasificación y presenta una convergencia más sana en su gráfica de `train_loss`. Los cambios de arquitectura y activación de los Runs C y D demuestran leves fluctuaciones al modificar la forma en que la red traza la frontera de decisión.

In [11]:
# 1. Ejecutar 5 semillas distintas para el Run C
for i in range(5):
    semilla_variada = SEED + i
    correr_experimento(f"Run_C_Seed_{semilla_variada}", True, (64, 32), 'relu', semilla_variada, "semillas")

# 2. Replicación exacta del original
correr_experimento("Run_C_Replica", True, (64, 32), 'relu', SEED, "replica")
print("Experimentos finalizados. Revisa la UI de MLflow para ver los resultados.")

🏃 View run Run_C_Seed_42 at: http://127.0.0.1:5000/#/experiments/1/runs/13e20e1775954561bd3f9ebaadc3d547
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run Run_C_Seed_43 at: http://127.0.0.1:5000/#/experiments/1/runs/2da31b6459f4459a9347603709ce3428
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run Run_C_Seed_44 at: http://127.0.0.1:5000/#/experiments/1/runs/1c9881d5ca3a4c5f8dd75265722a17ba
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run Run_C_Seed_45 at: http://127.0.0.1:5000/#/experiments/1/runs/efc4fc7abef94853b636e1ca4990a69e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run Run_C_Seed_46 at: http://127.0.0.1:5000/#/experiments/1/runs/2c587149aefa4b49ae155b1628fce5e4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run Run_C_Replica at: http://127.0.0.1:5000/#/experiments/1/runs/71a3f95a38214a27b31f152c910aef6a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Experiment

In [12]:
# Exportar las ejecuciones a CSV
runs_df = mlflow.search_runs()
columnas = ['run_id', 'tags.mlflow.runName', 'tags.fase', 'params.escalado', 'params.arquitectura', 'params.activacion', 'params.random_state', 'metrics.acc_test', 'metrics.recall_maligno']
runs_df = runs_df[[c for c in columnas if c in runs_df.columns]]
runs_df.to_csv("runs.csv", index=False)
print("runs.csv guardado. Descárgalo desde el panel de archivos de la izquierda.")

# Cierra el túnel por seguridad
ngrok.kill()

runs.csv guardado. Descárgalo desde el panel de archivos de la izquierda.


### Preguntas de Cierre

1. **¿Qué cambió al escalar las variables?**
Las métricas y la convergencia mejoraron notablemente en el Run B respecto al Run A. El *feature scaling* (estandarización) impidió que las características de mayor orden de magnitud dominaran los gradientes de la red, evitando la saturación de las funciones de activación.

2. **¿Más capas y neuronas mejoraron el resultado?**
La mejora al pasar de (30,) a (64,32) no siempre es absoluta. Al evaluar la variación introducida solo por cambiar la semilla inicial, nos damos cuenta de que las diferencias pueden ser producto del azar (distribución inicial de pesos) y no necesariamente de que la arquitectura más profunda sea superior para este dataset.

3. **Con tus 5 semillas, ¿qué tan grande fue la variación?**
La exactitud y la sensibilidad oscilaron dependiendo exclusivamente del `random_state` del modelo. Si la magnitud de esta variación por azar es igual o mayor a la diferencia obtenida al cambiar la arquitectura (Run B vs C), implica que no podemos declarar que una configuración es estadísticamente "mejor" sin hacer múltiples réplicas y promediarlas.

4. **En el diagnóstico de cáncer, ¿qué error es más costoso y qué métrica priorizarías?**
El error más crítico es el **falso negativo** (decir que un tumor es benigno cuando en realidad es maligno), porque el paciente perdería acceso a tratamiento urgente. Por ende, la métrica principal a observar y maximizar es la **sensibilidad (recall) de la clase maligna**.

5. **¿Qué información de MLflow no tendrías si solo usaras print()?**
Se perdería la **observabilidad** estructurada. Un simple `print` no asocia las métricas a un ID de experimento, ni guarda los metadatos exactos de la versión del entorno. Tampoco tendría un registro histórico visual de la pérdida (`train_loss`) época tras época, ni guardaría los artefactos (imágenes PNG de las matrices de confusión) vinculados a su configuración específica. Para que sea replicable, además registraría las versiones exactas de las librerías (`scikit-learn`, `pandas`).